In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler , MinMaxScaler, RobustScaler


In [ ]:
df = pd.read_csv("cleaned_data_missing_outliers.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
for col in df.columns:
    unique_values = df[col].nunique()
    values = df[col].unique()
    print(f"{col}: {unique_values} unique values")
    print(f"Values: {values}\n")

In [ ]:


import seaborn as sns
import matplotlib.pyplot as plt

def show_histogram_with_stats(df, column_name):
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(5, 3))
    sns.histplot(df[column_name], kde=True, color="skyblue", bins=50, alpha=0.5)
    plt.axvline(df[column_name].mean(), color="red", linestyle="dashed", linewidth=1, label="Mean")
    plt.axvline(df[column_name].median(), color="green", linestyle="dashed", linewidth=1, label="Median")
    plt.legend()
    plt.title('Original ' + column_name.capitalize() + ' Distribution')
    plt.xlabel(column_name.capitalize())
    plt.ylabel('Count')
    plt.show()


- define y 
- train val test split 
- feature transformation  (scaling ,encoding , binning )   (numerical(scale) and categorical(encoding) features)
- feature extraction and engineering 
- feature selection (filter, wrapper, embedded methods)


# define our target 


In [ ]:
#split inot new col for the values of price_egp
show_histogram_with_stats(df, 'price_egp')


In [ ]:
# binning the price_egp using qcut to create 3 bins (0-100, 100-200, 200+)

df['price_egp_bin'] = pd.qcut(df['price_egp'], q=3, labels=[0,1,2])

In [ ]:
df['price_egp_bin'].value_counts()

# Split Data into Train and Test Sets   

In [ ]:
# split data into train and test sets
from sklearn.model_selection import train_test_split
X = df.drop(['price_egp', 'price_egp_bin'], axis=1)
# X = df.drop('price_category', axis=1)
y = df['price_egp_bin']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# split trian to train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:

X_train.loc[X_train["amenities"] == "No amenities listed", "amenities"] = ""
X_val.loc[X_val["amenities"] == "No amenities listed", "amenities"] = ""
X_train["amenities_count"] = (
    X_train["amenities"]
    .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
)

X_val["amenities_count"] = (
    X_val["amenities"]
    .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
)
X_test.loc[X_test["amenities"] == "No amenities listed", "amenities"] = ""
X_test["amenities_count"] = (   
    X_test["amenities"]
    .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
)


In [ ]:
X_train_raw=X_train.copy()  
X_val_raw=X_val.copy()
X_test_raw=X_test.copy()


In [ ]:
X_train=X_train_raw
X_val=X_val_raw
X_test=X_test_raw

# Feature Scaling 
-   standardization
-   Min-Max Scaling
-   Robust Scaling 

**Transform numerical features to a common scale.**

### lat, lon 
lat:
- mean 29.88
- std 0.66
- min 25
- max 30.99
- median 30.01

lon:
- mean 31.46
- std 0.69
- min 27.97
- max 34.89
- median 31.25

**will use StandardScaler**

In [ ]:
show_histogram_with_stats(X_train, 'lat')
show_histogram_with_stats(X_train, 'lon')

In [ ]:
# apply scaling 
scaler = RobustScaler()
X_train['lat'] = scaler.fit_transform(X_train[['lat']])
X_val['lat'] = scaler.transform(X_val[['lat']])
X_test['lat'] = scaler.transform(X_test[['lat']])
show_histogram_with_stats(X_train, 'lat')



scaler = RobustScaler()
X_train['lon'] = scaler.fit_transform(X_train[['lon']])
X_val['lon'] = scaler.transform(X_val[['lon']])
X_test['lon'] = scaler.transform(X_test[['lon']])
show_histogram_with_stats(X_train, 'lon')




###  (4) **area_value**
-   mean=145
-   median=145
-   Q1=116
-   Q3=173
-   max=765

**mean = median** <br>
**So normal distribution so will apply Standardization**

In [ ]:
show_histogram_with_stats(X_train, 'area_value')


In [ ]:

# apply scaling 
scaler = StandardScaler()
X_train['area_value'] = scaler.fit_transform(X_train[['area_value']])
X_val['area_value'] = scaler.transform(X_val[['area_value']])
X_test['area_value'] = scaler.transform(X_test[['area_value']])
#after
show_histogram_with_stats(X_train, 'area_value')



###  (6) **distance features**
- *dist_nearest_school_km*
- *dist_nearest_hospital_km*
- *dist_nearest_supermarket_km*
- *dist_nearest_mall_km*
- *dist_nearest_transit_station_km*
- *dist_nearest_cafe_restaurant_km*

right skewed <br>
**So will apply Robust Scaling**

In [ ]:
for col in X_train.columns:
    if col.startswith('dist_nearest'):
        show_histogram_with_stats(X_train, col)

In [ ]:

for col in X_train.columns:
    if col.startswith('dist_nearest'):
        scaler = RobustScaler()
        X_train[col] = scaler.fit_transform(X_train[[col]])
        X_val[col] = scaler.transform(X_val[[col]])
        X_test[col] = scaler.transform(X_test[[col]])
        show_histogram_with_stats(X_train, col)

###  (7) **Count features**
- *school_count_within_3km*
- *hospital_count_within_3km*
- *supermarket_count_within_3km*
- *mall_count_within_3km*
- *transit_station_count_within_3km*
- *cafe_restaurant_count_within_3km*

bounded values non negative. <br>
**So will apply min-max scaling**

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        show_histogram_with_stats(X_train, col)

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        scaler = MinMaxScaler()
        X_train[col] = scaler.fit_transform(X_train[[col]])
        X_val[col] = scaler.transform(X_val[[col]])
        X_test[col] = scaler.transform(X_test[[col]])
        show_histogram_with_stats(X_train, col)

In [ ]:
X_train_Scaled=X_train.copy()  
X_val_Scaled=X_val.copy()
X_test_Scaled=X_test.copy()


In [ ]:
X_train=X_train_Scaled
X_val=X_val_Scaled
X_test=X_test_Scaled

# Feature Encoding 
-   One-Hot Encoding
-   Label Encoding
-   Target Encoding
-   Binary Encoding
-   Frequency Encoding
-   Rare Encoding

### (1) **Ordinal encode**  listing_level 
-   *standard* = 0
-  *featured* = 1
-  *premium* = 2
-  *hot* = 3
-  *superhot* = 4

as there is a clear order will apply Label Encoding

In [ ]:
listing_map = {
    "standard": 0,
    "featured": 1,
    "premium": 2,
    "hot": 3,
    "superhot": 4
}

for df_ in [X_train, X_val, X_test]:
    df_["listing_level"] = df_["listing_level"].map(listing_map)

### (2) **completion_status**              
-   *under-construction*    
-   *off_plan*      
-   *completed*     


### (3) **furnished**
-   *Unfurnished*    
-   *Unknown*         
-   *Furnished*        
-   *PARTLY*

we will apply one-hot encoding for both completion_status and furnished as they are nominal categorical variables with no clear order.

In [ ]:
cat_cols = ["completion_status", "furnished"]

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# ALIGN COLUMNS (VERY IMPORTANT)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

### (4) Frequency encode 
-   *city*
-   *town*
-   *district*

In [ ]:
freq_cols = ["city", "town", "district"]

for col in freq_cols:
    freq_map = X_train[col].value_counts(normalize=True)

    for df in [X_train, X_val, X_test]:
        df[col] = df[col].map(freq_map).fillna(0)

        

### (5) amenities convert to binary features


In [ ]:

X_train['amenities'] = X_train['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)
X_val['amenities']   = X_val['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)
X_test['amenities']  = X_test['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)

train_amenities = X_train['amenities'].str.get_dummies(sep='|')
val_amenities   = X_val['amenities'].str.get_dummies(sep='|')
test_amenities  = X_test['amenities'].str.get_dummies(sep='|')

val_amenities  = val_amenities.reindex(columns=train_amenities.columns, fill_value=0)
test_amenities = test_amenities.reindex(columns=train_amenities.columns, fill_value=0)

X_train = pd.concat([X_train.drop(columns=['amenities']), train_amenities], axis=1)
X_val   = pd.concat([X_val.drop(columns=['amenities']),   val_amenities],   axis=1)
X_test  = pd.concat([X_test.drop(columns=['amenities']),  test_amenities],  axis=1)

###  **convert bool to  int**


In [ ]:
for df in [X_train, X_val, X_test]:
    bool_cols = df.select_dtypes(bool).columns
    df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
for col in df.columns:
    unique_values = df[col].nunique()
    values = df[col].unique()
    print(f"{col}: {unique_values} unique values")
    print(f"Values: {values}\n")

In [ ]:
X_train.columns

In [ ]:
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()


In [ ]:
X_train=X_train_encoded
X_val=X_val_encoded
X_test=X_test_encoded

# Discretization and Binning
-   Equal Width Binning
-   Equal Frequency Binning

**for some feature that is may be hard to model as continuous variable, we can try to discretize it into bins and convert it to categorical variable.**

In [ ]:
X_train=X_train_raw
X_val=X_val_raw
X_test=X_test_raw

## (1) Area
- min = 40
- max = 320
- Q1 = 121
- median = 150
- Q3 = 179

In [ ]:
area_bins = [0, 100, 150, 220, float("inf")]

for df_ in [X_train, X_val, X_test]:
    df_["area_bin"] = pd.cut(
        df_["area_value"],
        bins=area_bins,
        labels=[0, 1, 2, 3],
        include_lowest=True
    ).astype(float)

## (2) bedrooms
- min = 1
- max = 7
- median = 3
- most values = 2–3 bedrooms

In [ ]:
bedroom_bins = [0, 2, 4, float("inf")]

for df_ in [X_train, X_val, X_test]:
    df_["bedrooms_bin"] = pd.cut(
        df_["bedrooms"],
        bins=bedroom_bins,
        labels=[0, 1, 2],
        include_lowest=True
    ).astype(int)

## (3) bathrooms
- min = 1
- max = 8
- median = 2

In [ ]:
bathroom_bins = [0, 2, 4, float("inf")]

for df_ in [X_train, X_val, X_test]:
    df_["bathrooms_bin"] = pd.cut(
        df_["bathroom"],
        bins=bathroom_bins,
        labels=[0, 1, 2],
        include_lowest=True
    ).astype(int)

### (4) distance features

In [ ]:
distance_cols = [
    "dist_nearest_school_km",
    "dist_nearest_hospital_km",
    "dist_nearest_supermarket_km",
    "dist_nearest_mall_km",
    "dist_nearest_transit_station_km",
    "dist_nearest_cafe_restaurant_km"
]

distance_bins = [0, 1, 3, 7, float("inf")]

for col in distance_cols:
    for df_ in [X_train, X_val, X_test]:
        df_[f"{col}_bin"] = pd.cut(
            df_[col],
            bins=distance_bins,
            labels=[0, 1, 2, 3],
            include_lowest=True
        ).astype(float)

In [ ]:
count_cols = [
    "school_count_within_3km",
    "hospital_count_within_3km",
    "supermarket_count_within_3km",
    "mall_count_within_3km",
    "transit_station_count_within_3km",
    "cafe_restaurant_count_within_3km"
]

for col in count_cols:
    X_train[col + "_bin"] = pd.cut(
        X_train[col],
        bins=[-1, 0, 3, 10, float("inf")],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X_val[col + "_bin"] = pd.cut(
        X_val[col],
        bins=[-1, 0, 3, 10, float("inf")],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X_test[col + "_bin"] = pd.cut(
        X_test[col],
        bins=[-1, 0, 3, 10, float("inf")],
        labels=[0, 1, 2, 3]
    ).astype(int)

In [ ]:
bin_cols = [col for col in X_train.columns if col.endswith('_bin')]

for col in bin_cols:
    print(f"{col}: {sorted(X_train[col].unique())}")

In [ ]:
constant_bins = [col for col in bin_cols if X_train[col].std() == 0]
print("⚠️ Constant bin columns (drop these):", constant_bins)

In [ ]:
import matplotlib.pyplot as plt

original_cols = [col.replace('_bin', '') for col in bin_cols if col.replace('_bin', '') in X_train.columns]

for orig in original_cols:
    bin_col = orig + '_bin'
    print(f"\n--- {orig} ---")
    print(X_train.groupby(bin_col)[orig].agg(['min', 'max', 'mean', 'count']))

In [ ]:
print(X_train[bin_cols].isnull().sum()[lambda x: x > 0])

# Feature Interactions
-   Arithmetic Complications
-   Statistical Aggregations
-   Boolean and Logical Combinations


In [ ]:
X_train['luxury_score']= X_train[['private pool', "private garden", 'view of water', 'covered parking']].sum(axis=1)
X_train['bedroom_density']= X_train['bedrooms'] / (X_train['area_value'] + 1e-5) 
X_train['bathroom_density']= X_train['bathrooms'] / (X_train['area_value'] + 1e-5)
X_train['total_rooms']= X_train['bedrooms'] + X_train['bathrooms']
X_train['room_density']= X_train['total_rooms'] / (X_train['area_value'] + 1e-5)

X_train['accessibility_score']= X_train['school_count_within_3km'] + X_train['hospital_count_within_3km'] + X_train['supermarket_count_within_3km']+ X_train['transit_station_count_within_3km']
X_train['lifestyle_score'] = X_train['mall_count_within_3km '] + X_train['cafe_restaurant_count_within_3km']


# Feature Selection
-   Filter Methods
-   Wrapper Methods
-   pemutation importance

# Documentation
